In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv("/content/drive/MyDrive/Regression Competition files/train.csv")
test = pd.read_csv("/content/drive/MyDrive/Regression Competition files/test.csv")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def engineer_features(df):
    X = df.drop(columns=["Id", "SalePrice"], errors="ignore").copy()

    X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
    X["YearsSinceRemodel"] = X["YrSold"] - X["YearRemodAdd"]
    X["HouseAgeSquared"] = X["HouseAge"] ** 2

    X["LogLotArea"] = np.log1p(X["LotArea"])
    X["LogLivingArea"] = np.log1p(X["GrLivArea"])
    X["BathroomEquivalent"] = X["FullBath"] + 0.5 * X["HalfBath"]

    X["OutdoorArea"] = X[
        ["WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
    ].sum(axis=1, min_count=1)

    quality_scores = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

    for column in ["ExterQual", "KitchenQual", "HeatingQC"]:
        X[f"{column}Score"] = X[column].map(quality_scores)

    for column in ["MSSubClass", "YrSold", "MoSold"]:
        X[column] = X[column].astype(str)

    return X


X = engineer_features(train)
y = train["SalePrice"]
X_test = engineer_features(test)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)


In [ ]:

factor_blocks = {
    "land": ["LogLotArea", "LotFrontage"],
    "structure": [
        "LogLivingArea", "TotalBsmtSF", "GarageArea", "BathroomEquivalent"
    ],
    "quality": ["OverallQual", "ExterQualScore", "KitchenQualScore"],
    "depreciation": ["HouseAge", "HouseAgeSquared", "YearsSinceRemodel"],
    "condition": ["OverallCond", "HeatingQCScore"],
    "amenities": ["Fireplaces", "OutdoorArea", "PoolArea"]
}

categorical_columns = [
    "Neighborhood", "MSZoning", "MSSubClass", "YrSold", "MoSold"
]


In [ ]:

def factor_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("input_scaler", StandardScaler()),
        ("pca", PCA(n_components=1, svd_solver="full")),
        ("factor_scaler", StandardScaler())
    ])


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    [
        (name, factor_pipeline(), columns)
        for name, columns in factor_blocks.items()
    ]
    + [("categories", categorical_pipeline, categorical_columns)],
    remainder="drop"
)

model = Pipeline([
    ("factors", preprocessor),
    ("regression", Ridge(solver="lsqr", tol=1e-6, max_iter=10000))
])


In [ ]:

search = GridSearchCV(
    model,
    param_grid={
        "regression__alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000]
    },
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

search.fit(X_train, y_train)

holdout_predictions = search.best_estimator_.predict(X_valid)
holdout_rmse = mean_squared_error(y_valid, holdout_predictions) ** 0.5

print(f"Best alpha: {search.best_params_['regression__alpha']:g}")
print(f"RMSE: ${holdout_rmse:,.2f}")


Best alpha: 10
RMSE: $34,681.74


In [ ]:

final_model = clone(search.best_estimator_)
final_model.fit(X, y)

submission = pd.DataFrame({
    "Id": test["Id"].to_numpy(),
    "SalePrice": final_model.predict(X_test)
})

assert len(submission) == len(test)
assert submission["Id"].is_unique
assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] > 0).all()

submission.to_csv("the_winning.csv", index=False)


In [ ]:
official_test = pd.read_csv("/content/drive/MyDrive/Regression Competition files/test.csv")
sample = pd.read_csv("/content/drive/MyDrive/Regression Competition files/solns.csv")

test_ids = set(official_test["Id"])
sample_ids = set(sample["Id"])

print("Official test rows:", len(official_test))
print("Sample submission rows:", len(sample))

print("\nFirst test IDs:")
print(official_test["Id"].head(10).tolist())

print("\nFirst sample IDs:")
print(sample["Id"].head(10).tolist())

print("\nIDs in sample but not test:", len(sample_ids - test_ids))
print("IDs in test but not sample:", len(test_ids - sample_ids))

print("\nSame ID sets:", test_ids == sample_ids)
print(
    "Same IDs in the same order:",
    official_test["Id"].tolist() == sample["Id"].tolist()
)

Official test rows: 260
Sample submission rows: 260

First test IDs:
[1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210]

First sample IDs:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

IDs in sample but not test: 260
IDs in test but not sample: 260

Same ID sets: False
Same IDs in the same order: False


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

X = train.drop(columns=["Id", "SalePrice"], errors="ignore").copy()
y = train["SalePrice"]

if "MSSubClass" in X.columns:
    X["MSSubClass"] = X["MSSubClass"].astype(str)

numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        keep_empty_features=True
    ))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

categorical_mask = (
    [False] * len(numeric_columns)
    + [True] * len(categorical_columns)
)

boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", HistGradientBoostingRegressor(
        loss="squared_error",
        categorical_features=categorical_mask,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=False,
        random_state=42
    ))
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

boosting_search = GridSearchCV(
    boosting_pipeline,
    param_grid={
        "regression__max_leaf_nodes": [7, 15],
        "regression__min_samples_leaf": [10, 20],
        "regression__l2_regularization": [1, 10, 100]
    },
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

boosting_search.fit(X_train, y_train)

predictions = boosting_search.best_estimator_.predict(X_valid)

rmse = mean_squared_error(y_valid, predictions) ** 0.5
print(f"New RMSE: ${rmse:,.2f}")

New RMSE: $29,499.88


In [ ]:
from sklearn.base import clone

official_test = pd.read_csv("/content/drive/MyDrive/Regression Competition files/test.csv")
sample = pd.read_csv("/content/drive/MyDrive/Regression Competition files/solns.csv")

X_submission = engineer_features(official_test)

if "MSSubClass" in X_submission.columns:
    X_submission["MSSubClass"] = (
        X_submission["MSSubClass"].astype(str)
    )

assert set(X_submission.columns) == set(X.columns), (
    "Training and test predictor columns do not match."
)
X_submission = X_submission[X.columns]

final_boosting_model = clone(boosting_search.best_estimator_)
final_boosting_model.fit(X, y)

predicted_prices = final_boosting_model.predict(X_submission)

predictions = pd.DataFrame({
    "Id": sample["Id"].to_numpy(),
    "SalePrice": predicted_prices
})

assert len(predictions) == len(sample) == 260
assert predictions["Id"].is_unique
assert sample["Id"].is_unique
assert set(predictions["Id"]) == set(sample["Id"])

submission = sample[["Id"]].merge(
    predictions,
    on="Id",
    how="left",
    validate="one_to_one",
    sort=False
)

assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] > 0).all()

submission.to_csv("the_new_winning.csv", index=False)